In [16]:
import numpy as np
import scipy.stats as st
import torch
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score

from mixmil import MixMIL
from mixmil.data import load_data
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV

In [ ]:
N = 3_000
Q = 10
K = 3
P = 1


I = 300
C = N * I

F = torch.randn(N, K)
F = torch.cat([torch.ones(N, 1), F], dim=1)
alpha = torch.randn(K + 1, 1) * 0.01
Xs = [torch.randn(I, Q) for _ in range(N)]
gamma = torch.randn(Q, 1)
beta = torch.randn(Q, 1)

logits = torch.tensor([(torch.softmax(Xs[i] @ gamma, dim=0) * Xs[i]).sum(0) @ beta + F[i] @ alpha for i in range(N)])
logits = (logits - logits.mean(0)) / logits.std(0)
logits = logits + 3  # uncomment to make unbalanced


y = torch.bernoulli(torch.sigmoid(logits)).reshape(N, 1)
y.mean()


tensor(0.9407)

In [18]:
n_epochs = 250

In [19]:
def print_metrics(y, probs):
    auc = roc_auc_score(y, probs)
    ap = average_precision_score(y, probs)
    true_prev = y.mean().item()
    pred_prev = probs.mean().item()
    print(f"AUC: {auc:.4f}, AP: {ap:.4f}, true prev: {true_prev:.4f}, pred prev: {pred_prev:.4f}")

In [20]:
X_mean = torch.vstack([X.mean(0) for X in Xs]).numpy()
X_mean = (X_mean - X_mean.mean(0)) / (X_mean.std(0) * np.sqrt(X_mean.shape[1]))
X_lr = np.hstack([F.numpy(), X_mean])
lr = LogisticRegressionCV(Cs=10, cv=5, class_weight="balanced")
lr.fit(X_lr, y.ravel())
preds = lr.predict_proba(X_lr)[:, 1]
print_metrics(y, preds)

AUC: 0.5985, AP: 0.9602, true prev: 0.9407, pred prev: 0.5128


In [21]:
X_mean = torch.vstack([X.mean(0) for X in Xs]).numpy()
X_mean = (X_mean - X_mean.mean(0)) / (X_mean.std(0) * np.sqrt(X_mean.shape[1]))
X_lr = np.hstack([F.numpy(), X_mean])
lr = LogisticRegressionCV(Cs=10, cv=5)
lr.fit(X_lr, y.ravel())
preds = lr.predict_proba(X_lr)[:, 1]
print_metrics(y, preds)

AUC: 0.5719, AP: 0.9551, true prev: 0.9407, pred prev: 0.9407


In [22]:
model = MixMIL.init_with_mean_model(Xs, F, y, likelihood="binomial", n_trials=1)
preds = torch.sigmoid(F @ model.alpha.detach() + model.predict(Xs))
print_metrics(y, preds)
model.train(Xs, F, y, n_epochs=n_epochs)
preds = torch.sigmoid(F @ model.alpha.detach() + model.predict(Xs))
print_metrics(y, preds)

GLMM Init:   0%|          | 0/1 [00:00<?, ?it/s]

AUC: 0.5886, AP: 0.9585, true prev: 0.9407, pred prev: 0.9425


Epoch:   0%|          | 0/250 [00:00<?, ?it/s]

AUC: 0.7484, AP: 0.9779, true prev: 0.9407, pred prev: 0.9411


In [ ]:
p = y.float().mean(dim=0)  # per-task if needed
pos_weight = 2 * (1 - p) / (p + 1e-12)  # for BCEWithLogitsLoss, the two in frontis a bit arbitrary
model = MixMIL.init_with_mean_model(Xs, F, y, likelihood="binomial", n_trials=1, standardize="none")
# model.fit_scaling(Xs)
model.pos_weight = pos_weight
preds = torch.sigmoid(F @ model.alpha.detach() + model.predict(Xs))
print_metrics(y, preds)
model.train(Xs, F, y, n_epochs=n_epochs)
preds = torch.sigmoid(F @ model.alpha.detach() + model.predict(Xs))
print_metrics(y, preds)

GLMM Init:   0%|          | 0/1 [00:00<?, ?it/s]

AUC: 0.5707, AP: 0.9548, true prev: 0.9407, pred prev: 0.9428


Epoch:   0%|          | 0/250 [00:00<?, ?it/s]

AUC: 0.7142, AP: 0.9732, true prev: 0.9407, pred prev: 0.6705
